# AI-Publisher Docker Image Builder v2

Bu notebook, AI-Publisher platformu icin Docker imajlarini Google Colab'da Kaniko ile build eder.
Imajlar Google Drive'a `.tar.gz` olarak kaydedilir, yerel makinede `docker compose up` ile yuklenir.

### Degisiklikler (v2):
- **Video codec fix**: cogvideox, wan, ltx, hunyuan, svd, animatediff, wan25 -> MJPG -> libx264 + yuv420p
- **Colab runtime kalkti**: Bu notebook sadece build icin, runtime sunucu Docker container'larda
- **Force-Rebuild**: Video modelleri zorunlu rebuild edilir (codec fix icin)
- **Incremental**: Ses/efekt modelleri Drive'da varsa atlanir

---
Her hucreyi sirayla calistirin.

## Hucre 1: Runtime Kontrolu

GPU ve sistem durumunu kontrol et. Build asamasinda GPU gerekli degil (Kaniko CPU'da calisir),
ancak dogrulama sirasinda model cikarimi yapilacaksa T4 GPU onerilir.

In [ ]:
#@title Runtime Kontrolu { display-mode: "form" }
import subprocess
import os

def check_runtime():
    print("=" * 60)
    print("AI-Publisher Docker Build - Runtime Kontrolu")
    print("=" * 60)
    
    # GPU kontrolu
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            gpu_info = result.stdout.strip().split(", ")
            print(f"GPU: {gpu_info[0]}")
            print(f"   Toplam VRAM: {gpu_info[1]} MB")
            print(f"   Bos VRAM: {gpu_info[2]} MB")
        else:
            print("GPU bulunamadi. Build icin GPU gerekli degil, devam.")
    except FileNotFoundError:
        print("nvidia-smi bulunamadi. Build icin GPU gerekli degil, devam.")
    
    # Disk alani
    disk = subprocess.run(["df", "-h", "/content"], capture_output=True, text=True)
    if disk.returncode == 0:
        lines = disk.stdout.strip().split("\n")
        if len(lines) > 1:
            parts = lines[1].split()
            print(f"Disk: {parts[2]} kullanildi / {parts[1]} toplam")
    
    # RAM
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal"):
                total_kb = int(line.split()[1])
                print(f"RAM: {total_kb // 1024} MB")
                break
    
    print("\nRuntime hazir. Sonraki hucreye gecin.")
    return True

check_runtime()

## Hucre 2: Drive Mount ve Bagimliliklar

Google Drive'i bagla, Kaniko + registry + pigz kur.

In [ ]:
#@title Drive Mount ve Bagimliliklar { display-mode: "form" }
import subprocess
import sys
import os
import time

print("Bagimliliklar kuruluyor...")

# pip kontrol
subprocess.run("rm -f /var/lib/dpkg/lock-frontend /var/lib/apt/lists/lock /var/cache/apt/archives/lock 2>/dev/null || true", shell=True)
subprocess.run("dpkg --configure -a 2>/dev/null || true", shell=True)
r = subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True, text=True)
if r.returncode != 0:
    print("  pip bulunamadi, apt-get ile kuruluyor...")
    subprocess.run("apt-get update -q && apt-get install -y -q python3-pip", shell=True, check=True, timeout=120)

# Google Drive mount
try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
        print("Google Drive baglandi.")
    else:
        print("Google Drive zaten bagli.")
except Exception as e:
    print(f"Drive mount hatasi: {e}")

print("\nKaniko + Registry + pigz kurulumu...")

# Kaniko kurulumu
kaniko_checks = [
    {"path": "/kaniko/executor", "label": "Kaniko"},
    {"path": "/usr/local/bin/registry", "label": "Registry"},
    {"path": "/usr/local/bin/pigz", "label": "pigz"},
]
all_ok = all(os.path.exists(c["path"]) for c in kaniko_checks)

if not all_ok:
    KANIKO_SCRIPT = """
    #!/bin/bash
    set -e
    echo "  Kaniko indiriliyor..."
    if [ ! -f /kaniko/executor ]; then
        curl -L -o /kaniko/executor https://github.com/GoogleContainerTools/kaniko/releases/download/v1.23.2/kaniko-linux-amd64 2>/dev/null || \
        curl -L -o /kaniko/executor https://github.com/GoogleContainerTools/kaniko/releases/download/v1.21.1/kaniko-linux-amd64
        chmod +x /kaniko/executor
        ln -sf /kaniko/executor /usr/local/bin/kaniko
    fi
    
    echo "  Registry indiriliyor..."
    if [ ! -f /usr/local/bin/registry ]; then
        curl -L -o /tmp/registry.tar.gz https://github.com/distribution/distribution/releases/download/v2.8.3/registry_2.8.3_linux_amd64.tar.gz
        tar -xzf /tmp/registry.tar.gz -C /tmp/ registry
        cp /tmp/registry /usr/local/bin/registry
        rm -f /tmp/registry.tar.gz /tmp/registry
    fi
    
    echo "  pigz indiriliyor..."
    if ! command -v pigz &> /dev/null; then
        apt-get install -y -q pigz 2>/dev/null || \
        (curl -L -o /usr/local/bin/pigz https://zlib.net/pigz/pigz-2.8.tar.gz 2>/dev/null && chmod +x /usr/local/bin/pigz)
    fi
    
    # Registry yapilandirmasi
    mkdir -p /etc/docker/registry
    cat > /etc/docker/registry/config.yml << 'EOF'
version: 0.1
log:
  level: error
storage:
  filesystem:
    rootdirectory: /var/lib/registry
http:
  addr: :5000
EOF
    mkdir -p /var/lib/registry
    echo "  Kaniko + Registry + pigz hazir."
"""
    subprocess.run(["bash", "-c", KANIKO_SCRIPT], check=True, timeout=120)
    print("Kaniko, Registry ve pigz basariyla kuruldu.")
else:
    print("Kaniko, Registry ve pigz zaten mevcut.")

print("\nDrive mount ve bagimliliklar tamam. Sonraki hucreye gecin.")

## Hucre 3: Repo Klonlama

GitHub'dan en son kodu cek. Codec fix'in dahil oldugundan emin olmak icin `main` branch'ini kullan.

In [ ]:
#@title Repo Klonlama { display-mode: "form" }
import subprocess
import os
from google.colab import userdata

REPO_URL = "https://github.com/ahmetkasim/AI-Publisher.git"
WORK_DIR = "/content/AI-Publisher"

print("Repo klonlaniyor...")

# GitHub PAT - Colab secrets'dan al
GITHUB_PAT = ""
try:
    GITHUB_PAT = userdata.get("GITHUB_PAT")
    if GITHUB_PAT:
        print("Colab secrets'dan GITHUB_PAT alindi.")
except Exception:
    pass

if not GITHUB_PAT:
    # Colab form'dan almayi dene
    try:
        from IPython.display import display
        import ipywidgets as widgets
        pat_input = widgets.Password(description="GITHUB_PAT:", style={"description_width": "initial"})
        display(pat_input)
        print("\nGITHUB_PAT yok! Lutfen Colab Secrets'a ekleyin:")
        print("  SOLDAN: Anahtar simgesi -> Add new secret")
        print("  Name: GITHUB_PAT, Value: <github_pat>")
        print("  Notebook access: ON")
        print("  Sonra bu hucreyi yeniden calistirin.")
        raise ValueError("GITHUB_PAT gerekli")
    except ImportError:
        pass
    raise ValueError("GITHUB_PAT bulunamadi! Colab Secrets'a ekleyin.")

if os.path.exists(WORK_DIR):
    print("Repo zaten klonlanmis, guncelleniyor...")
    os.chdir(WORK_DIR)
    subprocess.run(["git", "fetch", "origin"], check=True, timeout=30)
    subprocess.run(["git", "reset", "--hard", "origin/main"], check=True, timeout=30)
    print("Repo guncellendi.")
else:
    auth_url = REPO_URL.replace("https://", f"https://{GITHUB_PAT}@")
    subprocess.run(["git", "clone", auth_url, WORK_DIR], check=True, timeout=60)
    os.chdir(WORK_DIR)
    print("Repo klonlandi.")

# Build script'inin var oldugunu dogrula
os.chdir(WORK_DIR + "/colab_docker")
assert os.path.exists("build_all.sh"), "build_all.sh bulunamadi!"
subprocess.run(["chmod", "+x", "build_all.sh"], check=True)
print(f"build_all.sh hazir. Commit: {subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout.strip()}")

## Hucre 4: Force-Rebuild Base + Video Modelleri

Video modelleri (codec fix iceren) zorunlu rebuild edilir.
Sirasiyla: base (altyapi), cogvideox, wan, ltx, hunyuan, svd, animatediff, wan25

> **Tahmini sure**: Base ~5dk, her video modeli ~15-30dk, toplam ~2 saat
> Colab baglantisi koparsa, bu hucreyi yeniden calistir (var olan imajlar atlanir)

In [ ]:
#@title Force-Rebuild: Base + Video Modelleri { display-mode: "form" }
import subprocess
import os
import time

os.chdir("/content/AI-Publisher/colab_docker")

VIDEO_MODELS = [
    "base",
    "cogvideox",
    "wan",
    "ltx",
    "hunyuan",
    "svd",
    "animatediff",
    "wan25",
]

print("Force-Rebuild basliyor: Video modelleri (codec fix ile)")
print(f"Modeller: {', '.join(VIDEO_MODELS)}")
print("=" * 60)

total_start = time.time()

# build_all.sh'i video modelleri ile cagir
result = subprocess.run(
    ["bash", "build_all.sh"] + VIDEO_MODELS,
    capture_output=False,
    text=True,
    timeout=72000  # 20 saat timeout
)

total_duration = (time.time() - total_start) / 60
print(f"\nVideo model build tamamlandi. Toplam sure: {total_duration:.1f} dk")
print(f"Cikis kodu: {result.returncode}")

## Hucre 5: Incremental Build - Ses/Efekt Modelleri

Ses ve efekt modelleri incremental build edilir. Drive'da varsa atlanir.
**Bagimsizdir** - video build'u beklemez, ayri oturumda calistirilabilir.

Modeller: xtts, audioldm2, wav2lip, musetalk, whisper, stablediffusion, kokorotts, f5tts, lora-trainer

In [ ]:
#@title Incremental Build: Ses/Efekt Modelleri { display-mode: "form" }
import subprocess
import os
import time

os.chdir("/content/AI-Publisher/colab_docker")

print("Incremental build basliyor: Ses/Efekt modelleri")
print("(Drive'da var olan imajlar atlanacak)")
print("=" * 60)

total_start = time.time()

result = subprocess.run(
    ["bash", "build_all.sh"],
    capture_output=False,
    text=True,
    timeout=43200  # 12 saat
)

total_duration = (time.time() - total_start) / 60
print(f"\nSes/Efekt model build tamamlandi. Toplam sure: {total_duration:.1f} dk")
print(f"Cikis kodu: {result.returncode}")

## Hucre 6: Imaj Dogrulama

Drive'daki imajlari kontrol et: boyut, tarih, sayi.
Video modellerinde codec fix'i dogrula (opsiyonel).

In [ ]:
#@title Imaj Dogrulama { display-mode: "form" }
import subprocess
import os
import json

DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks/docker/images"
MIN_SIZE = 100 * 1024 * 1024  # 100 MB

print("Drive'daki imajlar kontrol ediliyor...")
print("=" * 60)

if not os.path.exists(DRIVE_DIR):
    print(f"HATA: {DRIVE_DIR} bulunamadi! Drive mount edilmemis olabilir.")
else:
    import glob
    files = sorted(glob.glob(os.path.join(DRIVE_DIR, "*.tar.gz")))
    print(f"Toplam {len(files)} imaj dosyasi bulundu.\n")
    
    video_models = ["base", "cogvideox", "wan", "ltx", "hunyuan", "svd", "animatediff", "wan25"]
    audio_models = ["xtts", "audioldm2", "wav2lip", "musetalk", "whisper", "stablediffusion", "kokorotts", "f5tts", "lora-trainer"]
    
    # Model bazli durum
    all_models = video_models + audio_models
    drive_map = {os.path.basename(f).replace(".tar.gz", ""): f for f in files}
    
    for model in all_models:
        if model in drive_map:
            fpath = drive_map[model]
            size = os.path.getsize(fpath)
            size_mb = size / (1024 * 1024)
            mtime = os.path.getmtime(fpath)
            import datetime
            date_str = datetime.datetime.fromtimestamp(mtime).strftime("%Y-%m-%d %H:%M")
            status = "OK" if size >= MIN_SIZE else "KUCUK"
            print(f"  [{status:6s}] {model:20s} {size_mb:8.1f} MB  {date_str}")
        else:
            print(f"  [  EKSIK] {model:20s} Drive'da bulunamadi!")
    
    print()
    missing = [m for m in all_models if m not in drive_map]
    small = [m for m in all_models if m in drive_map and os.path.getsize(drive_map[m]) < MIN_SIZE]
    
    if not missing and not small:
        print("Tum imajlar basariyla build edilmis ve Drive'a kaydedilmis.")
    else:
        if missing:
            print(f"EKSIK: {', '.join(missing)}")
        if small:
            print(f"KUCUK/BOZUK: {', '.join(small)}")

print("\nDogrulama tamamlandi.")

## Hucre 7: Ozet ve Sonraki Adimlar

Build tamamlandiginda:
1. Imajlari yerel makineye indir: `colab_docker/pull_images.sh` (Drive'dan kopyala)
2. Docker compose ile baslat: `docker compose -f colab_docker/docker-compose.yml up -d`
3. Servisleri dogrula: `curl http://localhost:5001/health`

### Gerekli Portlar (localhost:5001-5016)
- 5001 cogvideox - Video (I2V)
- 5002 xtts - TTS (Turkce)
- 5003 audioldm2 - SFX
- 5004 wav2lip - Lip-Sync
- 5005 musetalk - Lip-Sync (alternatif)
- 5006 whisper - STT
- 5007 stablediffusion - Gorsel
- 5008 wan - Video (T2V)
- 5009 ltx - Video (I2V)
- 5010 hunyuan - Video
- 5011 kokorotts - TTS (hizli)
- 5012 svd - Video (I2V)
- 5013 animatediff - Video
- 5014 wan25 - Video (genis)
- 5015 f5tts - TTS
- 5016 lora-trainer - LoRA